In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/content/2015_100_skill_builders_main_problems.csv')

In [4]:
df.head()

,user_id,log_id,sequence_id,correct
0,50121,167478035,7014,0.0
1,50121,167478043,7014,1.0
2,50121,167478053,7014,1.0
3,50121,167478069,7014,1.0
4,50964,167478041,7014,1.0


In [8]:
class IRT2PL:
  def __init__(self, num_users, num_items):
    self.theta = np.random.normal(0, 1, num_users)
    self.a = np.ones(num_items)
    self.b = np.zeros(num_items)


  def sigmoid(self, x):
    return 1 / (1 + np.exp(-x))

  def predict(self, u, i):
    return self.sigmoid(self.a[i] * (self.theta[u] - self.b[i]))

  def train(self, data, lr = 0.01, epochs = 50):
    for epoch in range(epochs):
      total_loss = 0

      for u, i, r in data:
        p = self.predict(u, i)

        p = np.clip(p, 1e-6, 1 - 1e-6)

        error = r - p

        # gradients
        d_theta = self.a[i] * error
        d_b = -self.a[i] * error
        d_a = error * (self.theta[u] - self.b[i])

        # update
        self.theta[u] += lr * d_theta
        self.b[i] += lr * d_b
        self.a[i] += lr * d_a


        self.a[i] = np.clip(self.a[i], 0.01, 3)

        total_loss += -(r * np.log(p) + (1 - r) * np.log(1 - p))

      print(f"Epoch {epoch + 1}, Loss : {total_loss:.4f}")

In [11]:
import pandas as pd

df = pd.read_csv("/content/2015_100_skill_builders_main_problems.csv")

# keep only required columns FIRST
df = df[['user_id', 'sequence_id', 'correct']]

# convert to int
df['user_id'] = df['user_id'].astype(int)
df['sequence_id'] = df['sequence_id'].astype(int)
df['correct'] = df['correct'].astype(int)

# remap IDs (VERY IMPORTANT)
df['user_id'] = df['user_id'].astype('category').cat.codes
df['sequence_id'] = df['sequence_id'].astype('category').cat.codes

# now convert to numpy
data = df.values

# counts
num_users = df['user_id'].nunique()
num_items = df['sequence_id'].nunique()

In [12]:
model = IRT2PL(num_users, num_items)
model.train(data, lr=0.01, epochs = 100)

Epoch 1, Loss : 430642.5886
Epoch 2, Loss : 415474.5933
Epoch 3, Loss : 413139.6737
Epoch 4, Loss : 411978.3491
Epoch 5, Loss : 411223.5586
Epoch 6, Loss : 410655.5864
Epoch 7, Loss : 410186.9998
Epoch 8, Loss : 409775.6564
Epoch 9, Loss : 409399.8955
Epoch 10, Loss : 409045.0111
Epoch 11, Loss : 408700.8872
Epoch 12, Loss : 408359.2908
Epoch 13, Loss : 408016.5573
Epoch 14, Loss : 407670.0250
Epoch 15, Loss : 407315.5980
Epoch 16, Loss : 406952.6144
Epoch 17, Loss : 406580.8329
Epoch 18, Loss : 406200.0168
Epoch 19, Loss : 405808.9283
Epoch 20, Loss : 405408.7982
Epoch 21, Loss : 405000.5039
Epoch 22, Loss : 404582.3056
Epoch 23, Loss : 404157.2681
Epoch 24, Loss : 403726.9374
Epoch 25, Loss : 403292.4899
Epoch 26, Loss : 402854.8184
Epoch 27, Loss : 402415.1784
Epoch 28, Loss : 401975.6201
Epoch 29, Loss : 401537.7108
Epoch 30, Loss : 401101.6750
Epoch 31, Loss : 400668.5148
Epoch 32, Loss : 400238.3636
Epoch 33, Loss : 399812.7433
Epoch 34, Loss : 399392.3547
Epoch 35, Loss : 398976

In [13]:
print("User abilities:", model.theta)
print("Item difficulty:", model.b)
print("Item discrimination:", model.a)

User abilities: [-0.89084571  0.25535898 -0.04064288 ... -0.83235549 -0.93177603
  0.05163395]
Item difficulty: [-0.78382594 -1.3269723  -0.95907488 -1.27786304 -1.08773485 -2.45491887
 -0.70876802 -0.35907649 -0.22462816 -0.71559373 -1.21724722 -0.42662176
 -1.32255851 -0.54599464 -1.34363232 -1.65144079 -2.07025771 -1.50422478
 -1.49619589 -1.5704663  -3.03678754 -1.41398857 -2.38504516 -2.024852
 -1.90123663 -0.96030123 -2.31910877 -0.97265367 -1.13526478 -1.80668073
 -1.30488138 -2.15630605 -1.68709826 -1.28300203 -1.54623359 -2.20910008
 -1.91920246 -1.42723136 -1.73054528 -2.29104165 -1.74193238 -1.4971998
 -1.74818384 -1.30020422 -2.01996787 -1.62633699 -1.21119615 -2.80211786
 -1.27307653 -1.91069805 -0.75149332 -0.80855455  0.32174458 -2.29287201
 -1.76103731 -0.59161099 -0.81840149  0.34911269 -1.25157566 -0.3774864
 -0.65140748 -0.77313043  0.2202407  -0.25935055 -0.50254725 -0.90268134
 -0.39614652 -2.35667353 -0.20207747 -2.32050956 -1.74552714 -1.36631924
 -2.44732022 -1.

In [14]:
user = 0
item = 1

prob = model.predict(user, item)
print("Probability of correct:", prob)

Probability of correct: 0.586165631564927


In [15]:
def select_question(model, user, available_items):
    best_item = None
    best_info = -1

    theta = model.theta[user]

    for item in available_items:
        p = model.predict(user, item)
        info = (model.a[item] ** 2) * p * (1 - p)

        if info > best_info:
            best_info = info
            best_item = item

    return best_item

In [16]:
user = 0
available_items = list(range(num_items))

for step in range(5):
    q = select_question(model, user, available_items)

    print(f"Ask Question {q}")

    # simulate response (replace with real input)
    response = np.random.randint(0, 2)

    print("Response:", response)

    # update only theta (online learning)
    p = model.predict(user, q)
    error = response - p

    model.theta[user] += 0.01 * model.a[q] * error

    available_items.remove(q)

Ask Question 94
Response: 0
Ask Question 54
Response: 0
Ask Question 13
Response: 1
Ask Question 46
Response: 1
Ask Question 3
Response: 0


In [17]:
def evaluate(model, data):
    correct = 0

    for u, i, r in data:
        p = model.predict(u, i)
        pred = 1 if p > 0.5 else 0

        if pred == r:
            correct += 1

    return correct / len(data)

print("Accuracy:", evaluate(model, data))


Accuracy: 0.7253620008156573


In [18]:
from sklearn.metrics import roc_auc_score

y_true = []
y_pred = []

for u, i, r in data:
    u, i = int(u), int(i)
    p = model.predict(u, i)

    y_true.append(r)
    y_pred.append(p)

print("AUC:", roc_auc_score(y_true, y_pred))

AUC: 0.6998652375522365


In [21]:
print("a stats:")
print("min:", model.a.min(), "max:", model.a.max(), "mean:", model.a.mean())


a stats:
min: 0.06479931557542558 max: 1.7191267957419512 mean: 0.8268556443283163
